<a href="https://colab.research.google.com/github/saranoor/chatGPT/blob/master/preprocessing_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
! pip install wikipedia

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11695 sha256=80a263420c829d5abba299f24428e84582b9cb1b596ec8094244b69b18b76a45
  Stored in directory: /root/.cache/pip/wheels/07/93/05/72c05349177dca2e0ba31a33ba4f7907606f7ddef303517c6a
Successfully built wikipedia


In [8]:
!pip install transformers

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 84.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 KB 24.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 79.6 MB/s eta 0:00:00


In [78]:
import pandas as pd
import wikipedia

In [79]:
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

In [80]:
tokenizer.encode("how are you oding")

[4919, 389, 345, 267, 12083]

In [91]:
import re
from typing import Set
from transformers import GPT2TokenizerFast

import numpy as np
from nltk.tokenize import sent_tokenize

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

def count_tokens(text: str) -> int:
    """count the number of tokens in a string"""
    return len(tokenizer.encode(text))

def reduce_long(
    long_text: str, long_text_tokens: bool = True, max_len: int = 590
) -> str:
    """
    Reduce a long text to a maximum of `max_len` tokens by potentially cutting at a sentence end
    """
    if not long_text_tokens:
        long_text_tokens = count_tokens(long_text)
    if long_text_tokens > max_len:
        sentences = sent_tokenize(long_text.replace("\n", " "))
        ntokens = 0
        for i, sentence in enumerate(sentences):
            ntokens += 1 + count_tokens(sentence)
            if ntokens > max_len:
                return ". ".join(sentences[:i][:-1]) + "."

    return long_text

In [92]:
import pandas as pd
df_raw=pd.read_csv('')

In [93]:
df_raw[0:15]

,title,heading,content
0,10Pearls Policies and Procedures,Our Mission,TenPearls (Private) Limited is founded on the ...
1,10Pearls Policies and Procedures,Definitions,In this Manual the following words and express...
2,11Pearls Policies and Procedures,General,This Manual is designed to serve as the basic ...
3,10Pearls Policies and Procedures,Duties and Responsibilities,The following duties are expected to be perfor...
4,10Pearls Policies and Procedures,Employment Association with Tenpearls,Employees may be employed by the Company as ei...
5,10Pearls Policies and Procedures,Joining the Company,"On joining the company, the Employee must prov..."
6,10Pearls Policies and Procedures,Appointment and Probation,APPOINTING AUTHORITY\nThe appointing authority...
7,10Pearls Policies and Procedures,Monetary and Non-Monetary Benefits,The Company offers various Monetary and Non-Mo...
8,10Pearls Policies and Procedures,Leave Entitlement and Rule,This policy has been framed bearing in mind th...


In [100]:
def calculate_tokens(c, title, h, max_len):
  ncontent_ntokens = count_tokens(c)+ count_tokens(h)

    # Create a tuple of (title, section_name, content, number of tokens)
  if ncontent_ntokens<max_len:
    outputs=(title, h, c, ncontent_ntokens) 
  else:
    outputs=(title, h, reduce_long(c, max_len), count_tokens(reduce_long(c,max_len)))
  return outputs[3]

In [101]:
calculate_tokens("Karachi is a good city", "Karachi","Intro to karachi",590)

12

In [102]:
pd.set_option('max_columns', None)
df_raw['tokenized']=df_raw.apply(lambda x: calculate_tokens(x.content,x.title, x.heading, 590), axis=1)

In [103]:
df_raw

,title,heading,content,tokenized
0,10Pearls Policies and Procedures,Our Mission,TenPearls (Private) Limited is founded on the ...,100
1,10Pearls Policies and Procedures,Definitions,In this Manual the following words and express...,470
2,11Pearls Policies and Procedures,General,This Manual is designed to serve as the basic ...,241
3,10Pearls Policies and Procedures,Duties and Responsibilities,The following duties are expected to be perfor...,692
4,10Pearls Policies and Procedures,Employment Association with Tenpearls,Employees may be employed by the Company as ei...,546
5,10Pearls Policies and Procedures,Joining the Company,"On joining the company, the Employee must prov...",273
6,10Pearls Policies and Procedures,Appointment and Probation,APPOINTING AUTHORITY\nThe appointing authority...,441
7,10Pearls Policies and Procedures,Monetary and Non-Monetary Benefits,The Company offers various Monetary and Non-Mo...,1821
8,10Pearls Policies and Procedures,Leave Entitlement and Rule,This policy has been framed bearing in mind th...,131


In [105]:
df_raw.to_csv('tokenized_policy_doc.csv')